In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## EDA - Basic Exploration, Checking variables, Data types and Summary Statistics

In [ ]:
import matplotlib.pylab as plt
import lightgbm as lgb
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    KFold,
    StratifiedKFold,
    GroupKFold,
    StratifiedGroupKFold
)

In [ ]:
df=pd.read_csv("/kaggle/input/credit-risk-dataset/credit_risk_dataset.csv", encoding='latin')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## Memory Consumption Check

In [ ]:
df.memory_usage(deep=True)

In [ ]:
cat_cols = [col for col in df.columns if df[col].dtype =='O']
cat_cols

### Let's check unique value counts in categorical variable

In [ ]:
for col in cat_cols:
    print("------------------")
    print("Column Name {0} : Total Unique Values : {1} ".format(col,len(df[col].value_counts())))
    print(df[col].value_counts())
    print("------------------")
    

### Let's change the data type of column to Categorical

In [ ]:
for col in cat_cols:
    df[col] = df[col].astype('category')

In [ ]:
df.memory_usage(deep=True)

## Memory Consumption Achieved

### Lets look at the distribution of output variable

In [ ]:
df['loan_status'].value_counts(normalize=True)

In [ ]:
%matplotlib inline
# Creating Bar chart as the Target variable is Categorical
GroupedData=df.groupby('loan_status').size()
GroupedData.plot(kind='bar', figsize=(4,3))

### Though we have onl 80:20 ratio, we can still go ahead

## Duplicate data management

In [ ]:
df[df.duplicated()]

In [ ]:
print("Total Duplicated Values in dataframe are {0}".format(df[df.duplicated()].shape[0]))

### Let's confirm duplicate values if they are really fact

In [ ]:
df.query("person_age==43 & person_income==11340 &\
person_home_ownership=='RENT' & loan_intent=='EDUCATION'")

## Let's drop the duplicated values

In [ ]:
df.drop_duplicates(inplace=True)

## Numerical Variable Analysis

In [ ]:
df.describe()

### We could see, person_age max value is 144 Let's visualize it with boxplot

In [ ]:
import seaborn as sns
plt.figure(figsize=(5,5))
boxplot = sns.boxplot(y="person_age", data=df)

In [ ]:
df.query("person_age > 80")

### We can cap age above 84

In [ ]:
df = df.query("person_age < 85")

In [ ]:
boxplot = sns.boxplot(y="person_age", data=df)

## Lets Look at employee Length

In [ ]:
boxplot = sns.boxplot(y="person_emp_length", data=df)

### We could see one value at the top, seems to be outlier, lets confirm that

In [ ]:
df.query("person_emp_length > 60")

## Lets fill this value with nan and we would impute it later as only employee length seems to be wrognly entered

In [ ]:
filter_con = df["person_emp_length"] > 60
df.loc[filter_con, 'person_emp_length'] = np.nan

In [ ]:
boxplot = sns.boxplot(y="person_emp_length", data=df)

In [ ]:
df.head()

In [ ]:
35000 / 59000

### Loan Percent income seems to be direct calculation of percent of salary, let's confirm

In [ ]:
df["confirm_col_loan_percent_income"] = df['loan_amnt']/df['person_income']

In [ ]:
df = df.round({"confirm_col_loan_percent_income":2}) 

In [ ]:
df[df['confirm_col_loan_percent_income']!=df['loan_percent_income']]

### Through query, we could confirm that, its not the straigh calculation and we can't drop the values

## Imputing Missing values

In [ ]:
import missingno as msno
msno.matrix(df, figsize = (8,5));

In [ ]:
msno.bar(df, color = 'y', figsize = (8,5));

In [ ]:
df.isna().sum()

## The values are missing completely Random, we would use IterativeImputer to fill the missing value
## Fature Creation and Selection

### Lets Visualize person income by age

In [ ]:
plt.figure(figsize=(15,15))
boxplot = sns.boxplot(x="person_age", y="person_income", data=df)
boxplot.axes.set_title("Agewise Income", fontsize=16)
boxplot.set_xlabel("Age", fontsize=14)
boxplot.set_ylabel("Income", fontsize=14)
plt.show()

### The Salaries seems to be normal, only concern on 63 age, lets check

In [ ]:
df.query("person_age == 63")

### 1782000 seems to be a outlier, but we will still keep it for now

In [ ]:
df.query("person_income > 1500000")

### These records again seems to be a outliers, we will keep them as its as of now,
### We would apply capping technique to check if our model achieves any improvement

In [ ]:
df.head()

### Let's look at person income by person home owenrship

In [ ]:
plt.figure(figsize=(15,15))
boxplot = sns.boxplot(x="person_home_ownership", y="person_income", data=df)
boxplot.axes.set_title("Owenrship Income", fontsize=16)
boxplot.set_xlabel("Home Ownership", fontsize=14)
boxplot.set_ylabel("Income", fontsize=14)
plt.show()

### 

In [ ]:
df.query("person_income == 63")

In [ ]:
df['loan_amnt'].plot(kind='hist',color='g')

In [ ]:
35000/54400

In [ ]:
cat_cols = [col for col in df.columns if df[col].dtype == "category" and col != 'loan_status']
num_cols = [col for col in df.columns if df[col].dtype != "category" and col != 'loan_status']

In [ ]:
df.head(10)

## Let's create new Features

### 1. Higher Salary People
### Lets first group employee salary by age, then we will calculate upper whisker by calculating Q3 + (1.5) * IQR of that group
### For each employee, if that salary is above higher whisker then mark him as high salary person

In [ ]:
def get_higher_whisker(grp):
    try:
        q1, q3 = grp.quantile(0.25), grp.quantile(0.75)
        iqr = q3 - q1
        higher_whisker = q3 + (1.5*iqr)
        return higher_whisker
    except Exception as e:
        print("------------Error Occured-----------")
        print(traceback.format_exc())

In [ ]:
%%time
df["higher_whisker"] = df.groupby('person_age')['person_income'].transform(lambda x: get_higher_whisker(x))

In [ ]:
%%time
df['higher_salary'] = df['person_income'] > df['higher_whisker']

In [ ]:
pd.crosstab(df['loan_status'],df['higher_salary'])

### 2. Home owners would be bit of higher earning person or financial wel to do person.

In [ ]:
%%time
df['home_owner'] = df['person_home_ownership']=='OWN'

In [ ]:
pd.crosstab(df['loan_status'],df['home_owner'])

### 3. Employee having long services tend to be well to do 

In [ ]:
df['long_working'] = df['person_emp_length'] > 10

In [ ]:
pd.crosstab(df['loan_status'],df['long_working'])

### 4. Employee having very low loan requirement, tend to get loan approved easily

In [ ]:
df['loan_percent_calc'] = (df['loan_amnt'] / df['person_income'])*100
df['lower_loan_requirement'] =  df['loan_percent_calc'] < 20

In [ ]:
pd.crosstab(df['loan_status'],df['lower_loan_requirement'])

In [ ]:
df['higher_loan_requirement'] =  df['loan_percent_calc'] > 40

In [ ]:
pd.crosstab(df['loan_status'],df['higher_loan_requirement'])

In [ ]:
cat_cols.append(['higher_salary','home_owner','long_working','lower_loan_requirement','higher_loan_requirement'])

In [ ]:
cat_cols = ['person_home_ownership',
 'loan_intent',
 'loan_grade',
 'cb_person_default_on_file',
 'higher_salary',
  'home_owner',
  'long_working',
  'lower_loan_requirement',
  'higher_loan_requirement']

## Check Statistical Significant Relation with Output variable ( Feature Selection )

## Corelation of Categorical Predictor Vs Categorical Output Variable 
### Categorical Vs Categorical variable relation will be tested by Box Plot interpretation

In [ ]:
fig, plt_convas=plt.subplots(nrows=len(cat_cols), ncols=1, figsize=(10,30))
zipped_cols = zip(cat_cols, range(len(cat_cols)))
for col,i in zipped_cols:
    crosstab_loan_status = pd.crosstab(index=df[col], columns=df['loan_status'])
    crosstab_loan_status.plot.bar(color=['red','green'], ax=plt_convas[i], title=col+' Vs '+' Loan Status')

### Box Plot interpration
### 1. Grouped barplots generally shows the frequencies of category against output variables, on the X axis there is categories and Y axis shows frequencies.
### 2. If the ratio of bars is similar across all categories, then the two columns are not correlated.
### 3. We will confirm this with Chi Square test

In [ ]:
from scipy.stats import chi2_contingency
cols_selected = []
for col in cat_cols:
    crosstab_loan_status = pd.crosstab(index=df[col], columns=df['loan_status'])
    chi_square_res = chi2_contingency(crosstab_loan_status)
    if(chi_square_res[1] < 0.05):
        print(col," is Corealted with Loan Status variable.")
        cols_selected.append(col)

cols_selected

### As per aboove results, we can select person_home_ownership, loan_intent, loan_grade, cb_person_default_on_file variables can be considered for modelling, as they have good corelation
## Corelation of Contineous Predictor Vs Categorical Output Variable 
### Lets now check for numerical variable correlation with output variable 
### 1. We will first check the box plot visualiazation and then we will use ANOVA to confirm our findings

In [ ]:
fig, plot_canvas = plt.subplots(nrows=1, ncols=len(num_cols), figsize=(18,5))

for col , i in zip(num_cols, range(len(num_cols))):
    df.boxplot(column=col, by='loan_status', figsize=(5,5), vert=True, ax=plot_canvas[i])

### Lets confirm with ANOVA

In [ ]:
from scipy.stats import f_oneway
cols_selected = []
for col in num_cols:
    datagrouped = df.groupby("loan_status")[col].apply(list)
    anova_res = f_oneway(*datagrouped)
    print(col,':', anova_res[1])
    if(anova_res[1] < 0.05):
        print(col," is Corealted with Loan Status variable.")
        cols_selected.append(col)
    else:
        print(anova_res)

cols_selected

## Feature Selection Summary
### We have finally selected person_home_ownership, loan_intent, loan_grade, cb_person_default_on_file as Categorical Features
### We have finally selected 'person_age','person_income','loan_amnt','loan_percent_income','cb_person_cred_hist_length' as Numerical features

## Lets define our strategy for Classification ML Process
<ul>
    <li><b>Read Train Dataframe</b></li>
    
    <li><b>Convert all the cateogorical values datatype to category</b></li>
    <li><b>Drop Duplicate</b></li>
    <li><b>Get age only below 80</b></li>
    <li><b>Cap person_emp_length to only 60 and put nan for all aove values</b></li>
    <li><b>Impute Missing Values for numerical variable</b></li>
    <li><b>Scale numerical variable.</b></li>
    <li><b>Encode all categorical variable.</b></li>
    <li><b>Apply model and plot confusion matrix. </b></li>
    <li><b>Check balenced Accuracy , plot Learning curve and ROC curve </b></li>
    <li><b>Hypertune the model, cmpare each model and finalize</b></li>
</ul>

In [2]:
from sklearn.model_selection import ( train_test_split, KFold, StratifiedKFold,cross_val_score, RepeatedKFold, RandomizedSearchCV,learning_curve, ShuffleSplit, GridSearchCV)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import (IterativeImputer, SimpleImputer, KNNImputer)
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (AdaBoostRegressor, RandomForestClassifier, RandomForestRegressor)
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ( confusion_matrix , ConfusionMatrixDisplay, balanced_accuracy_score, roc_auc_score)
import plotly.graph_objects as go
import re
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from lightgbm import LGBMClassifier
import matplotlib.pylab as plt
np.seterr(divide = 'ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [3]:
class SelectColumnsTransformer():
    #The parameter "columns" is set and only those columns will be resulted 
    def __init__(self, columns=None):
        self.columns = columns
        
    #Transformer function will copy only those columns which has better corelation with output variable
    def transform(self, X, **transform_params):
        cpy_df = X[self.columns].copy()
        return cpy_df

    def fit(self, X, y=None, **fit_params):
        return self

In [4]:
class DataframeFunctionTransformer():
    def __init__(self, func):
        self.func = func

    def transform(self, input_df, **transform_params):
        return self.func(input_df)

    def fit(self, X, y=None, **fit_params):
        return self

In [5]:
class ml_support():
    def __init__(self):
        self.df = pd.read_csv("data/credit_risk_dataset.csv", encoding='latin')
        self.output_var = 'loan_status'
        self.num_cols = ['person_age','person_income','loan_amnt','loan_percent_income','cb_person_cred_hist_length']
        self.cat_cols = ['person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file','higher_salary','home_owner','long_working','lower_loan_requirement','higher_loan_requirement']
        # These are columns without feature engineering
        self.cols_wofeature = [col for col in self.df.columns if col != self.output_var ]
        # These are coluns after feature engineering
        self.cols_wfeature = ['person_age','person_income','loan_amnt','loan_percent_income','cb_person_cred_hist_length','person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file','higher_salary','home_owner','long_working','lower_loan_requirement','higher_loan_requirement']
        self.X = self.df[self.cols_wofeature]
        #Output variable
        self.y = self.df[self.output_var]
        self.random_state = 42
    
    def drop_duplicate(self):
        
        self.df.drop_duplicates(inplace=True)
        return self.df
    
    def get_train_test_data(self):
        
        #Split traininig and testing set
        X_train, X_test , y_train, y_test = train_test_split(self.X,self.y,test_size=0.2,
                                                             random_state = self.random_state, shuffle=True, stratify=self.df[self.output_var])
        return X_train, X_test , y_train, y_test
    
    def get_column_transformer(self,is_estimator, iterative_Estimator, is_TreeBased=False):
        
        #Pipineline created for munging numerical columns
        num_pipeline_steps = []
        iterative_imputer = IterativeImputer()
        if is_estimator:
            iterative_imputer = IterativeImputer(estimator=iterative_Estimator)
        
        num_pipeline_steps.append(('num_missing',iterative_imputer))
        
        if not is_TreeBased:
            num_pipeline_steps.append(('num_smoothening',PowerTransformer()))
        
        num_pipeline = Pipeline(num_pipeline_steps)
        
        #Pipineline created for categorical column encoding
        cat_pipeline = Pipeline([
            ('cat_encoding',OneHotEncoder(sparse=False,drop='if_binary',handle_unknown='ignore'))
        ])
        
        # Column Transformer is created which will call pipelines
        ct = ColumnTransformer([
            ('num_munging',num_pipeline,self.num_cols),
            ('cat_munging',cat_pipeline,self.cat_cols)
        ])
        return ct
    
    def get_final_pipeline(self,regression_model,columntransformer, feature_engieered):
    
        X_train, X_test , y_train, y_test  = self.get_train_test_data()
        features_pipeline = [feature_engieered]
        features_pipeline.append(("selector", SelectColumnsTransformer(self.cols_wfeature)))
        features_pipeline.append(('munging',columntransformer))
        features_pipeline.append(('model',regression_model))
        finalized_pipeline = Pipeline(features_pipeline)
        return finalized_pipeline;
    
    def predit_with_pipeline(self,pipeline,X_train,X_test,y_train):
        pipeline.fit(X_train,y_train)
        preds = pipeline.predict(X_test)
        return preds
    
    def get_best_params(self,pipeline,params):
    
        rscv = RandomizedSearchCV(pipeline, params, scoring='balanced_accuracy',
                              n_jobs=-1, n_iter=4, cv=5, random_state=self.random_state, verbose=3)
        rscv.fit(self.X,self.y)
        return rscv.best_params_

    def get_best_params_gcv(self,pipeline,params):
    
        rscv = GridSearchCV(pipeline, params, scoring='balanced_accuracy',
                              n_jobs=-1, cv=5, verbose=3)
        rscv.fit(self.X,self.y)
        return rscv.best_params_
    
    def plot_confusion_matrix(self,y_test, preds, finalized_pipeline):
    
        cm = confusion_matrix(y_test, preds, labels=finalized_pipeline.classes_)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels= finalized_pipeline.classes_)
        disp.plot();
        plt.show();
        
    def update_performance_matrics(self,model_info,df):
        
        filter_con =  df["Model_Name"] == model_info['Model_Name']
        if ((filter_con)).any():
            df.loc[filter_con, 'Score'] = model_info['Score']
        else:
            df = df.append(model_info, ignore_index=True)
        return df
    
    def plot_learning_curves(self,estimator):
        """
        Don't forget to change the scoring and plot labels
        based on the metric that you are using.
        """
        train_sizes, train_scores, test_scores = learning_curve(
            estimator=estimator,
            X=self.X,
            y=self.y,
            train_sizes=np.linspace(0.1, 1.0, 5),
            cv=5,
            scoring="balanced_accuracy",
            random_state=self.random_state,
            n_jobs=-1
        )
        train_mean = np.mean(train_scores, axis=1)
        test_mean = np.mean(test_scores, axis=1)
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=train_sizes,
                y=train_mean,
                name="Training Accuracy",
                mode="lines",
                line=dict(color="blue"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=train_sizes,
                y=test_mean,
                name="Validation Accuracy",
                mode="lines",
                line=dict(color="green"),
            )
        )
        fig.update_layout(
            title="Learning Curves",
            xaxis_title="Number of training examples",
            yaxis_title="Balenced Accuracy",
        )
        fig.show()

In [6]:
def get_higher_whisker(grp):
    try:
        q1, q3 = grp.quantile(0.25), grp.quantile(0.75)
        iqr = q3 - q1
        higher_whisker = q3 + (1.5*iqr)
        return higher_whisker
    except Exception as e:
        print("------------Error Occured-----------")
        print(traceback.format_exc())
        
def impute_higher_salary(df):
    
    df["higher_whisker"] = df.groupby('person_age')['person_income'].transform(lambda x: get_higher_whisker(x))
    df['higher_salary'] = df['person_income'] > df['higher_whisker']
    return df;

def define_home_owner(df):
    df['home_owner'] = df['person_home_ownership']=='OWN'
    return df

def define_loan_percentage(df):
    df['loan_percent_calc'] = (df['loan_amnt'] / df['person_income'])*100
    return df

def define_lower_loan_requirement(df):
    df['lower_loan_requirement'] =  df['loan_percent_calc'] < 20
    return df

def define_higher_loan_requirement(df):
    df['higher_loan_requirement'] =  df['loan_percent_calc'] > 40
    return df

def define_long_employement(df):
    df['long_working'] = df['person_emp_length'] > 10
    return df

def impute_empty_to_wrong_age(df):
        
    filter_con = df["person_age"] > 85
    df.loc[filter_con, 'person_age'] = np.nan
    return df

def impute_empty_to_wrong_employee_len(df):

    filter_con = df["person_emp_length"]  > 60
    df.loc[filter_con, 'person_emp_length'] = np.nan
    return df

def add_features(df):
    df = impute_empty_to_wrong_age(df)
    df = impute_empty_to_wrong_employee_len(df)
    df = impute_higher_salary(df)
    df = define_home_owner(df)
    df = define_loan_percentage(df)
    df = define_lower_loan_requirement(df)
    df = define_higher_loan_requirement(df)
    df = define_long_employement(df)
    return df

In [7]:
feature_engieered = ("FeatureEngineering_add_features", DataframeFunctionTransformer(add_features))

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = LogisticRegression(n_jobs=-1,max_iter=1000)
ct = ml_support_obj.get_column_transformer(False, "", False)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
df_modelPerformance = pd.DataFrame(columns=['Model_Name','Model_Pipeline','Score'])
dict_Baseline_Score = {
                "Model_Name" : "Baseline Logistic Regression",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
regression_model = LogisticRegression(n_jobs=-1)
params = {"munging__num_munging__num_missing__estimator": [LinearRegression(), RandomForestRegressor(random_state=0),KNeighborsRegressor()],
          "model__penalty":["l1", "l2", "none"],
          "model__C" : np.logspace(-4, 4, 20),
          "model__solver": ["newton-cg","lbfgs","liblinear","sag","saga"],
          "model__max_iter" : [100, 1000,2500, 5000]
        }
ct = ml_support_obj.get_column_transformer(False, "")
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
best_params = ml_support_obj.get_best_params(finalized_pipeline,params)

In [ ]:
best_params

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = LogisticRegression(n_jobs=-1,solver='sag',
                                      penalty='none',C=1438.44988828766,
                                      max_iter=2500)
ct = ml_support_obj.get_column_transformer(True, LinearRegression(n_jobs=-1))
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Hyper Tuned Logistic Regression",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = DecisionTreeClassifier(random_state=0)
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Baseline Decision Tree Logistic",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = DecisionTreeClassifier(random_state=42)
params = {"munging__num_munging__num_missing__estimator": [LinearRegression(n_jobs=-1), RandomForestRegressor(random_state=42),KNeighborsRegressor(n_jobs=-1)],
          "model__criterion":["gini", "entropy","log_loss"],
          "model__splitter" : ["best", "random"],
          "model__max_depth" : range(1,10),
          "model__min_samples_split": range(1,10),
          "model__min_samples_leaf": range(1,5),
          "model__max_features":["auto","sqrt","log2",None]
         }
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
best_params = ml_support_obj.get_best_params(finalized_pipeline,params)

In [ ]:
best_params

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = DecisionTreeClassifier(random_state=42,splitter='best',
                                         min_samples_split=8,min_samples_leaf=4,
                                         max_features='auto',max_depth=9,criterion='gini')
ct = ml_support_obj.get_column_transformer(True, KNeighborsRegressor(n_jobs=-1), True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Hypertuned Decision Tree Logistic",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = RandomForestClassifier(random_state = 42)
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Baseline Radnom Forest Classifier",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = RandomForestClassifier(random_state = 42)
# Number of trees in random forest
n_estimators = [int(x) for x in range(200,2000,200)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Method of selecting samples for training each tree
bootstrap = [True, False]
regression_model = RandomForestClassifier(random_state = 42)
params = {"munging__num_munging__num_missing__estimator": [LinearRegression(n_jobs=-1), RandomForestRegressor(random_state=42),KNeighborsRegressor(n_jobs=-1)],
          "model__n_estimators": n_estimators,
          "model__max_features": max_features,
          "model__max_depth": max_depth,
          "model__min_samples_split": min_samples_split,
          "model__min_samples_leaf": min_samples_leaf,
          "model__bootstrap": bootstrap
          }
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
best_params = ml_support_obj.get_best_params(finalized_pipeline,params)

In [ ]:
best_params

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = RandomForestClassifier(random_state = 42,n_estimators=1200,
                                          min_samples_split=5, min_samples_leaf = 1,
                                          max_features = 'auto', max_depth = None,
                                          bootstrap =True
                                         )
ct = ml_support_obj.get_column_transformer(True, KNeighborsRegressor(n_jobs=-1), True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Hypertuned RandomForest Tree Logistic",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
from lightgbm import LGBMClassifier
regression_model = LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Baseline LGBMClassifier Tree Logistic",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
params = {"munging__num_munging__num_missing__estimator": [LinearRegression(n_jobs=-1), RandomForestRegressor(random_state=42),KNeighborsRegressor(n_jobs=-1)],
          "model__n_estimators":  [int(x) for x in range(200,2000,200)],
          "model__learning_rate": [0.001,0.01,0.1,1,10],
          "model__boosting_type": ['gbdt','goss','dart'],
          }
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
best_params = ml_support_obj.get_best_params(finalized_pipeline,params)

In [ ]:
best_params

In [ ]:
%%time
ml_support_obj = ml_support()
regression_model = LGBMClassifier(class_weight='balanced', random_state=42, 
                                  n_jobs=-1,n_estimators=200,learning_rate=0.001,
                                  boosting_type='dart'
                                 )
ct = ml_support_obj.get_column_transformer(True, LinearRegression(n_jobs=-1), False)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [ ]:
%%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Baseline Decision Tree Logistic",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)

In [ ]:
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)

In [ ]:
%%time
ml_support_obj.plot_learning_curves(finalized_pipeline)

In [ ]:
finalized_pipeline

In [ ]:
dict_Baseline_Score = {
                "Model_Name" : "Hypertuned LGBMClassifier",
                "Model_Pipeline" : [finalized_pipeline],
                "Score": balanced_accuracy_score(y_test, preds)
               }
df_modelPerformance = ml_support_obj.update_performance_matrics(dict_Baseline_Score,df_modelPerformance)
df_modelPerformance.sort_values('Score',ascending=False)


In [ ]:
model_pipeline = df_modelPerformance.loc[df_modelPerformance['Model_Name']=='Baseline LGBMClassifier Tree Logistic','Model_Pipeline'].values

In [ ]:
model_pipeline

In [ ]:
test=pd.read_csv("/kaggle/input/titanic/test.csv", encoding='latin')
submission_preds = model_pipeline[0][0].predict(test)
test_ids = test['PassengerId']
df_submit = pd.DataFrame({"PassengerId": test_ids.values,
                         "Survived" : submission_preds
                        })
df_submit.to_csv("submission.csv",index=False)

In [8]:
# %%time
# Adaboost (Boosting of multiple Decision Trees)
from sklearn.ensemble import AdaBoostRegressor
ml_support_obj = ml_support()
best_decision_tree_model = RandomForestClassifier(random_state = 42,n_estimators=1200,
                                          min_samples_split=5, min_samples_leaf = 1,
                                          max_features = 'auto', max_depth = None,
                                          bootstrap =True
                                         )
regression_model = AdaBoostRegressor()
ct = ml_support_obj.get_column_transformer(False, "", True)
X_train, X_test , y_train, y_test = ml_support_obj.get_train_test_data()
finalized_pipeline = ml_support_obj.get_final_pipeline(regression_model,ct,feature_engieered)

In [9]:
# %%time
preds = ml_support_obj.predit_with_pipeline(finalized_pipeline,X_train,X_test,y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [10]:
# fix (option 2)--------preds should be categorical as y_test
preds = np.where(preds >= 0.5, 1, 0)
print("Balenced Accuracy Score : {0}".format(balanced_accuracy_score(y_test, preds)))

Balenced Accuracy Score : 0.7168414747090788


In [ ]:
ml_support_obj.plot_confusion_matrix(y_test, preds, finalized_pipeline)